# ROBUST04 Ranking Competition - BM25 Baseline

**Final Project Part A - Text Retrieval and Search Engines**

This notebook implements a BM25 baseline for document ranking on ROBUST04.

**Pipeline:**
1. Load ROBUST04 index via Pyserini
2. Parse queries and qrels
3. Tune BM25 parameters on 50 train queries
4. Generate rankings for all 249 queries
5. Output TREC format for submission

## 1. Configuration

In [0]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# File paths - UPDATE THESE FOR YOUR DBFS PATHS
QUERIES_PATH = "queriesROBUST.txt"
QRELS_PATH = "qrels_50_Queries"
OUTPUT_DIR = "runs/"

# BM25 parameters to tune
BM25_K1_RANGE = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
BM25_B_RANGE = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

# Default parameters (will be updated after tuning)
BM25_K1_DEFAULT = 0.9
BM25_B_DEFAULT = 0.4

# Retrieval configuration
TOP_K = 1000  # Number of documents to retrieve per query
RUN_TAG = "bm25_baseline"

# Parallel processing
MAX_WORKERS = 20

print("✅ Configuration loaded")

✅ Configuration loaded


## 2. Install Dependencies

In [0]:
# Install Pyserini (version compatible with Java 17)
%pip install pyserini==0.21.0 -q

# Restart Python to pick up new packages
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 3. Java Setup

In [0]:
import subprocess
import os

# Set Java environment variables
os.environ['JAVA_HOME'] = '/usr/lib/jvm/zulu17-ca-amd64'
os.environ['JVM_PATH'] = '/usr/lib/jvm/zulu17-ca-amd64/lib/server/libjvm.so'

# Verify Java installation
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print("Java version:")
print(result.stderr)

Java version:
openjdk version "1.8.0_462"
OpenJDK Runtime Environment (Zulu 8.88.0.19-CA-linux64) (build 1.8.0_462-b08)
OpenJDK 64-Bit Server VM (Zulu 8.88.0.19-CA-linux64) (build 25.462-b08, mixed mode)



## 4. Load ROBUST04 Index

In [0]:
%pip install numpy==1.24.3 -q
%pip install faiss-cpu==1.7.4 -q
%pip install pyserini==0.21.0 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
petastorm 0.12.1 requires pyspark>=2.1.0, which is not installed.
faiss-cpu 1.13.2 requires numpy<3.0,>=1.25.0, but you have numpy 1.24.3 which is incompatible.
ydata-profiling 4.5.1 requires numpy<1.24,>=1.16.0, but you have numpy 1.24.3 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader

# Load index
print("Loading ROBUST04 index...")
searcher = LuceneSearcher.from_prebuilt_index('robust04')
index_reader = IndexReader.from_prebuilt_index('robust04')

print(f"✅ Index loaded successfully!")
print(f"📊 Index stats: {index_reader.stats()}")

2026-01-04 18:30:11.098356: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-04 18:30:12.411924: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading ROBUST04 index...


lucene-index.robust04.20221005.252b5e.tar.gz: 1.68GB [00:17, 106MB/s]                            


✅ Index loaded successfully!
📊 Index stats: {'total_terms': 174540872, 'documents': 528030, 'non_empty_documents': 528030, 'unique_terms': 923436}


## 5. Load Queries and Qrels

In [0]:
import re
from collections import defaultdict

def load_queries(queries_path):
    """Parse queriesROBUST.txt file."""
    queries = {}
    
    with open(queries_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Format: "query_id<tab>query_text"
            parts = line.split('\t', 1)
            if len(parts) == 2:
                qid = parts[0].strip()
                query_text = parts[1].strip()
                queries[qid] = query_text
    
    return queries

def load_qrels(qrels_path):
    """
    Parse qrels file.
    Format: query_id 0 doc_id relevance
    """
    qrels = defaultdict(dict)
    
    with open(qrels_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                doc_id = parts[2]
                relevance = int(parts[3])
                qrels[qid][doc_id] = relevance
    
    return qrels

# Load data
print("Loading queries...")
all_queries = load_queries(QUERIES_PATH)
print(f"✅ Loaded {len(all_queries)} queries")

print("\nLoading qrels (train)...")
qrels = load_qrels(QRELS_PATH)
train_qids = list(qrels.keys())
print(f"✅ Loaded qrels for {len(train_qids)} train queries")

# Split train/test
train_queries = {qid: all_queries[qid] for qid in train_qids if qid in all_queries}
test_qids = [qid for qid in all_queries.keys() if qid not in train_qids]
test_queries = {qid: all_queries[qid] for qid in test_qids}

print(f"\n📊 Dataset split:")
print(f"   Train: {len(train_queries)} queries")
print(f"   Test:  {len(test_queries)} queries")
print(f"   Total: {len(all_queries)} queries")

# Show sample
print(f"\n🔍 Sample queries:")
for qid in list(train_queries.keys())[:3]:
    print(f"   {qid}: {train_queries[qid]}")

Loading queries...
✅ Loaded 249 queries

Loading qrels (train)...
✅ Loaded qrels for 50 train queries

📊 Dataset split:
   Train: 50 queries
   Test:  199 queries
   Total: 249 queries

🔍 Sample queries:
   301: international organized crime
   302: poliomyelitis post polio
   303: hubble telescope achievements


## 6. BM25 Retrieval Functions

In [0]:
def retrieve_bm25(query_text, k1=0.9, b=0.4, k=1000):
    """
    Retrieve documents using BM25.
    
    Args:
        query_text: Query string
        k1: BM25 k1 parameter
        b: BM25 b parameter  
        k: Number of documents to retrieve
        
    Returns:
        List of (doc_id, score) tuples
    """
    searcher.set_bm25(k1=k1, b=b)
    hits = searcher.search(query_text, k=k)
    
    results = [(hit.docid, hit.score) for hit in hits]
    return results

def batch_retrieve(queries_dict, k1=0.9, b=0.4, k=1000, desc="Retrieving"):
    """
    Retrieve documents for multiple queries.
    
    Returns:
        Dict mapping query_id -> [(doc_id, score), ...]
    """
    from tqdm import tqdm
    
    results = {}
    for qid, query_text in tqdm(queries_dict.items(), desc=desc):
        results[qid] = retrieve_bm25(query_text, k1=k1, b=b, k=k)
    
    return results

print("✅ BM25 retrieval functions ready")

✅ BM25 retrieval functions ready


## 7. MAP Evaluation

In [0]:
def calculate_ap(ranked_list, qrels_for_query):
    """
    Calculate Average Precision for a single query.
    
    Args:
        ranked_list: List of doc_ids in ranked order
        qrels_for_query: Dict mapping doc_id -> relevance (0 or 1)
        
    Returns:
        Average Precision score
    """
    if not qrels_for_query:
        return 0.0
    
    num_relevant = sum(1 for rel in qrels_for_query.values() if rel > 0)
    if num_relevant == 0:
        return 0.0
    
    num_relevant_retrieved = 0
    precision_sum = 0.0
    
    for i, doc_id in enumerate(ranked_list, start=1):
        if doc_id in qrels_for_query and qrels_for_query[doc_id] > 0:
            num_relevant_retrieved += 1
            precision_at_i = num_relevant_retrieved / i
            precision_sum += precision_at_i
    
    if num_relevant_retrieved == 0:
        return 0.0
    
    return precision_sum / num_relevant

def calculate_map(retrieval_results, qrels):
    """
    Calculate Mean Average Precision across all queries.
    
    Args:
        retrieval_results: Dict mapping qid -> [(doc_id, score), ...]
        qrels: Dict mapping qid -> {doc_id: relevance}
        
    Returns:
        MAP score
    """
    ap_scores = []
    
    for qid in retrieval_results:
        if qid not in qrels:
            continue
        
        ranked_list = [doc_id for doc_id, _ in retrieval_results[qid]]
        ap = calculate_ap(ranked_list, qrels[qid])
        ap_scores.append(ap)
    
    if not ap_scores:
        return 0.0
    
    return sum(ap_scores) / len(ap_scores)

print("✅ MAP evaluation functions ready")

✅ MAP evaluation functions ready


## 8. Test Baseline Performance

In [0]:
print("🧪 Testing baseline BM25 (k1=0.9, b=0.4) on train set...\n")

# Run baseline
baseline_results = batch_retrieve(
    train_queries, 
    k1=BM25_K1_DEFAULT, 
    b=BM25_B_DEFAULT,
    k=TOP_K,
    desc="Baseline retrieval"
)

# Calculate MAP
baseline_map = calculate_map(baseline_results, qrels)

print(f"\n{'='*60}")
print(f"📊 BASELINE RESULTS")
print(f"{'='*60}")
print(f"BM25 Parameters: k1={BM25_K1_DEFAULT}, b={BM25_B_DEFAULT}")
print(f"MAP on train: {baseline_map:.4f}")
print(f"{'='*60}")

🧪 Testing baseline BM25 (k1=0.9, b=0.4) on train set...



Baseline retrieval: 100%|██████████| 50/50 [00:05<00:00,  9.79it/s]


📊 BASELINE RESULTS
BM25 Parameters: k1=0.9, b=0.4
MAP on train: 0.2455


## 9. Grid Search Parameter Tuning

In [0]:
from itertools import product
import time

def grid_search_bm25(queries_dict, qrels, k1_range, b_range, top_k=1000):
    """
    Grid search over BM25 parameters.
    
    Returns:
        Tuple of (best_k1, best_b, best_map, results_df)
    """
    import pandas as pd
    from tqdm import tqdm
    
    results = []
    best_map = 0
    best_k1 = None
    best_b = None
    
    param_combinations = list(product(k1_range, b_range))
    
    print(f"🔍 Grid searching {len(param_combinations)} parameter combinations...\n")
    
    for k1, b in tqdm(param_combinations, desc="Grid search"):
        # Retrieve documents
        retrieval_results = batch_retrieve(
            queries_dict, 
            k1=k1, 
            b=b, 
            k=top_k,
            desc=f"k1={k1:.1f}, b={b:.1f}"
        )
        
        # Calculate MAP
        map_score = calculate_map(retrieval_results, qrels)
        
        results.append({
            'k1': k1,
            'b': b,
            'MAP': map_score
        })
        
        if map_score > best_map:
            best_map = map_score
            best_k1 = k1
            best_b = b
    
    results_df = pd.DataFrame(results).sort_values('MAP', ascending=False)
    
    return best_k1, best_b, best_map, results_df

In [0]:
# Run grid search
print(f"Starting parameter tuning on {len(train_queries)} train queries...")
start_time = time.time()

best_k1, best_b, best_map, tuning_results = grid_search_bm25(
    train_queries,
    qrels,
    BM25_K1_RANGE,
    BM25_B_RANGE,
    top_k=TOP_K
)

elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"🎯 PARAMETER TUNING RESULTS")
print(f"{'='*60}")
print(f"Best parameters: k1={best_k1}, b={best_b}")
print(f"Best MAP: {best_map:.4f}")
print(f"Time elapsed: {elapsed:.1f}s")
print(f"{'='*60}\n")

print("Top 10 parameter combinations:")
display(tuning_results.head(10))

Starting parameter tuning on 50 train queries...
🔍 Grid searching 49 parameter combinations...



k1=0.8, b=0.2:  34%|███▍      | 17/50 [00:01<00:02, 16.37it/s]
k1=0.8, b=0.2:  38%|██

*** WARNING: max output size exceeded, skipping output. ***

Grid search: 100%|██████████| 49/49 [02:46<00:00,  3.40s/it]



🎯 PARAMETER TUNING RESULTS
Best parameters: k1=0.6, b=0.4
Best MAP: 0.2475
Time elapsed: 166.6s

Top 10 parameter combinations:


k1,b,MAP
0.6,0.4,0.24746374986866862
0.7,0.4,0.24738369066984195
0.8,0.4,0.2464183845920022
0.6,0.3,0.24632849238297932
0.7,0.3,0.24560009152105633
0.9,0.4,0.24546562746858008
0.6,0.5,0.2452845269085188
0.8,0.3,0.2452126794962819
0.6,0.2,0.24515213038724007
0.7,0.2,0.2451328351592288


In [0]:
print(tuning_results)

     k1    b       MAP
2   0.6  0.4  0.247464
9   0.7  0.4  0.247384
16  0.8  0.4  0.246418
1   0.6  0.3  0.246328
8   0.7  0.3  0.245600
23  0.9  0.4  0.245466
3   0.6  0.5  0.245285
15  0.8  0.3  0.245213
0   0.6  0.2  0.245152
7   0.7  0.2  0.245133
22  0.9  0.3  0.244954
14  0.8  0.2  0.244826
21  0.9  0.2  0.244823
30  1.0  0.4  0.244797
24  0.9  0.5  0.244198
10  0.7  0.5  0.244086
37  1.1  0.4  0.243938
17  0.8  0.5  0.243865
31  1.0  0.5  0.243801
28  1.0  0.2  0.243745
36  1.1  0.3  0.243350
29  1.0  0.3  0.243176
35  1.1  0.2  0.242961
38  1.1  0.5  0.242890
44  1.2  0.4  0.242652
43  1.2  0.3  0.242574
45  1.2  0.5  0.242416
4   0.6  0.6  0.242231
32  1.0  0.6  0.241901
42  1.2  0.2  0.241620
25  0.9  0.6  0.241497
18  0.8  0.6  0.241474
11  0.7  0.6  0.241416
39  1.1  0.6  0.240767
46  1.2  0.6  0.239996
5   0.6  0.7  0.239793
12  0.7  0.7  0.239646
19  0.8  0.7  0.239214
26  0.9  0.7  0.239065
33  1.0  0.7  0.238182
6   0.6  0.8  0.237734
13  0.7  0.8  0.237537
20  0.8  0.

## 10. TREC Output Format

In [0]:
def save_trec_format(retrieval_results, output_path, run_tag):
    """
    Save results in TREC format.
    
    Format: qid Q0 doc_id rank score run_tag
    """
    with open(output_path, 'w') as f:
        for qid in sorted(retrieval_results.keys(), key=int):
            ranked_docs = retrieval_results[qid]
            
            for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
                line = f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_tag}\n"
                f.write(line)
    
    print(f"✅ Saved TREC format: {output_path}")

print("✅ TREC output function ready")

✅ TREC output function ready


## 11. Generate Run 1 - Tuned BM25

In [0]:
import os

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 Generating Run 1: Tuned BM25 (k1={best_k1}, b={best_b})")
print(f"   Processing all {len(all_queries)} queries...\n")

# Retrieve for all queries
all_results = batch_retrieve(
    all_queries,
    k1=best_k1,
    b=best_b,
    k=TOP_K,
    desc="Run 1 retrieval"
)

# Save TREC format
run1_path = os.path.join(OUTPUT_DIR, "run_1.res")
save_trec_format(all_results, run1_path, f"bm25_k1_{best_k1}_b_{best_b}")

# Verify output
print(f"\n📊 Run 1 Statistics:")
print(f"   Total queries: {len(all_results)}")
print(f"   Avg docs per query: {sum(len(docs) for docs in all_results.values()) / len(all_results):.1f}")

# Show sample output
print(f"\n🔍 Sample output (first 5 lines):")
with open(run1_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"   {line.strip()}")

🚀 Generating Run 1: Tuned BM25 (k1=0.6, b=0.4)
   Processing all 249 queries...



Run 1 retrieval: 100%|██████████| 249/249 [00:18<00:00, 13.68it/s]


✅ Saved TREC format: runs/run_1.res

📊 Run 1 Statistics:
   Total queries: 249
   Avg docs per query: 969.2

🔍 Sample output (first 5 lines):
   301 Q0 FBIS4-40260 1 7.026700 bm25_k1_0.6_b_0.4
   301 Q0 FBIS4-46734 2 7.014100 bm25_k1_0.6_b_0.4
   301 Q0 FBIS3-24145 3 7.005000 bm25_k1_0.6_b_0.4
   301 Q0 FBIS4-39330 4 6.997100 bm25_k1_0.6_b_0.4
   301 Q0 FBIS4-7811 5 6.982200 bm25_k1_0.6_b_0.4


## 12. Validation Summary

In [0]:
# Calculate MAP on train set with tuned parameters
print("📊 Validating tuned parameters on train set...\n")

train_results_tuned = {qid: all_results[qid] for qid in train_queries.keys()}
train_map_tuned = calculate_map(train_results_tuned, qrels)

print(f"{'='*60}")
print(f"📈 FINAL VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Method: BM25")
print(f"Parameters: k1={best_k1}, b={best_b}")
print(f"MAP on train (50 queries): {train_map_tuned:.4f}")
print(f"Baseline MAP: {baseline_map:.4f}")
print(f"Improvement: {((train_map_tuned - baseline_map) / baseline_map * 100):+.2f}%")
print(f"{'='*60}")

📊 Validating tuned parameters on train set...

📈 FINAL VALIDATION RESULTS
Method: BM25
Parameters: k1=0.6, b=0.4
MAP on train (50 queries): 0.2475
Baseline MAP: 0.2455
Improvement: +0.81%


## 13. Analysis: Per-Query Performance

In [0]:
import pandas as pd

# Calculate AP for each train query
per_query_results = []

for qid in train_queries.keys():
    ranked_list = [doc_id for doc_id, _ in train_results_tuned[qid]]
    ap = calculate_ap(ranked_list, qrels[qid])
    
    per_query_results.append({
        'query_id': qid,
        'query_text': train_queries[qid],
        'AP': ap,
        'num_relevant': sum(1 for rel in qrels[qid].values() if rel > 0),
        'num_retrieved': len(ranked_list)
    })

df_analysis = pd.DataFrame(per_query_results).sort_values('AP')

print("📊 Per-Query Analysis\n")
print("Worst 5 performing queries:")
display(df_analysis.head(5))

print("\nBest 5 performing queries:")
display(df_analysis.tail(5))

print(f"\nAP Statistics:")
print(df_analysis['AP'].describe())

📊 Per-Query Analysis

Worst 5 performing queries:


query_id,query_text,AP,num_relevant,num_retrieved
309,rap crime,0.0010582010582010583,3,1000
305,dangerous vehicles,0.0013538880410983059,35,1000
314,marine vegetation,0.0019074645511326662,44,1000
322,international art crime,0.002334267040149393,34,1000
318,best retirement country,0.006486620619776333,113,1000



Best 5 performing queries:


query_id,query_text,AP,num_relevant,num_retrieved
302,poliomyelitis post polio,0.6237394055011076,65,1000
328,pope beatifications,0.7366666666666666,8,1000
312,hydroponics,0.7485234321921628,11,21
313,magnetic levitation maglev,0.7714913273769108,93,1000
348,agoraphobia,0.8099999999999999,5,6



AP Statistics:
count    50.000000
mean      0.247464
std       0.238773
min       0.001058
25%       0.054545
50%       0.158095
75%       0.363266
max       0.810000
Name: AP, dtype: float64


In [0]:
print(df_analysis.columns)
print(df_analysis[['query_id', 'query_text', 'AP', 'num_relevant']])

Index(['query_id', 'query_text', 'AP', 'num_relevant', 'num_retrieved'], dtype='object')
   query_id                        query_text        AP  num_relevant
8       309                         rap crime  0.001058             3
4       305                dangerous vehicles  0.001354            35
13      314                 marine vegetation  0.001907            44
21      322           international art crime  0.002334            34
17      318           best retirement country  0.006487           113
42      343                     police deaths  0.011725           258
14      315     unexplained highway accidents  0.022979            67
19      320        undersea fiber optic cable  0.026894             6
45      346             educational standards  0.030295           105
0       301     international organized crime  0.034488           448
43      344                     abuses e mail  0.035746             5
35      336                black bear attacks  0.043698            12
4

## 14. Summary & Next Steps

In [0]:
print(f"""
{'='*60}
🎉 BM25 BASELINE COMPLETE
{'='*60}

📊 Results Summary:
   • Method: BM25 with parameter tuning
   • Best parameters: k1={best_k1}, b={best_b}
   • Train MAP: {train_map_tuned:.4f}
   • Baseline improvement: {((train_map_tuned - baseline_map) / baseline_map * 100):+.2f}%

📁 Output Files:
   • {run1_path}

📈 Next Steps:
   1. Review per-query performance to identify failure modes
   2. Implement Method 2: Query Likelihood Dirichlet (QLD)
   3. Implement Method 3: BM25 + Cross-Encoder Reranking
   4. Compare all methods and select best for submission

💡 Tips for Improvement:
   • Analyze queries with AP < 0.1 - what went wrong?
   • Consider query expansion for short queries
   • Try ensemble methods (combine BM25 + QLD)
   • Implement reranking with cross-encoders

{'='*60}
""")


🎉 BM25 BASELINE COMPLETE

📊 Results Summary:
   • Method: BM25 with parameter tuning
   • Best parameters: k1=0.6, b=0.4
   • Train MAP: 0.2475
   • Baseline improvement: +0.81%

📁 Output Files:
   • runs/run_1.res

📈 Next Steps:
   1. Review per-query performance to identify failure modes
   2. Implement Method 2: Query Likelihood Dirichlet (QLD)
   3. Implement Method 3: BM25 + Cross-Encoder Reranking
   4. Compare all methods and select best for submission

💡 Tips for Improvement:
   • Analyze queries with AP < 0.1 - what went wrong?
   • Consider query expansion for short queries
   • Try ensemble methods (combine BM25 + QLD)
   • Implement reranking with cross-encoders


